In [1]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123",
)

In [2]:
import os

def upload_folder(local_folder, bucket, prefix=""):
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)

            # create object key (preserve folder structure)
            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(prefix, relative_path).replace("\\", "/")

            s3.upload_file(local_path, bucket, s3_key)
            print(f"Uploaded: {s3_key}")

In [3]:
upload_folder(
    local_folder="./bronze/",
    bucket="bronze",
    prefix=""   # optional: e.g. "raw/2026/"
)

Uploaded: all_stocks_5yr.csv


In [4]:
upload_folder(
    local_folder="./silver/",
    bucket="silver",
    prefix=""   # optional: e.g. "raw/2026/"
)

Uploaded: cleaned_stocks_data.csv/.part-00000-194655fd-7ca5-438c-b026-64ba490aae0a-c000.csv.crc
Uploaded: cleaned_stocks_data.csv/.part-00000-781137af-b9e6-453c-bc9e-83fa029d75a8-c000.csv.crc
Uploaded: cleaned_stocks_data.csv/._SUCCESS.crc
Uploaded: cleaned_stocks_data.csv/part-00000-194655fd-7ca5-438c-b026-64ba490aae0a-c000.csv
Uploaded: cleaned_stocks_data.csv/part-00000-781137af-b9e6-453c-bc9e-83fa029d75a8-c000.csv
Uploaded: cleaned_stocks_data.csv/_SUCCESS
Uploaded: stocks_after_fe.parquet/.part-00000-5d32381a-fb02-40cb-ac9b-4231d9376a74-c000.snappy.parquet.crc
Uploaded: stocks_after_fe.parquet/._SUCCESS.crc
Uploaded: stocks_after_fe.parquet/part-00000-5d32381a-fb02-40cb-ac9b-4231d9376a74-c000.snappy.parquet
Uploaded: stocks_after_fe.parquet/_SUCCESS


In [5]:
upload_folder(
    local_folder="./models/",
    bucket="silver",
    prefix=""   # optional: e.g. "raw/2026/"
)

Uploaded: ma_model.joblib


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("MinIO Upload") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()


:: loading settings :: url = jar:file:/home/ezz/myenv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ezz/.ivy2/cache
The jars for the packages stored in: /home/ezz/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-671690ac-a1d6-461d-ad8c-061e52ba4941;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 209ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted||

In [3]:
df = spark.read.csv("./bronze/all_stocks_5yr.csv", header=True)

In [4]:
df.write \
  .mode("overwrite") \
  .csv("s3a://bronze/my-data/")

26/05/02 21:01:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/02 21:01:58 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/02 21:01:58 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/02 21:01:58 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/02 21:01:58 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/02 21:01:58 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/02 21:01:58 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/02 21:01:58